# 🌸 17-Class Flower CNN Classifier — High Accuracy (Colab GPU)

This notebook builds a production-ready 17-class flower classifier. It:
- combines the original `train` and `validation` images,
- creates a **file-level stratified 80/10/10 split**,
- trains a strong CNN (EfficientNetB2 transfer learning),
- fine-tunes the CNN,
- evaluates on a held-out test set,
- saves `final_model.keras` and the exact `class_names.json`,
- provides a single-image predictor that returns one of the 17 allowed classes.

Allowed classes:
`bluebell, buttercup, colts_foot, cowslip, crocus, daffodil, daisy, dandelion, fritillary, iris, lily_valley, pansy, snowdrop, sunflower, tigerlily, tulip, windflower`

In [ ]:
# 1) Setup
!pip -q install -U tensorflow

import os, json, random, shutil, zipfile
from pathlib import Path
import numpy as np
import pandas as pd
import tensorflow as tf
import matplotlib.pyplot as plt

from sklearn.model_selection import train_test_split
from sklearn.utils.class_weight import compute_class_weight
from sklearn.metrics import classification_report, confusion_matrix
from PIL import Image

SEED = 42
random.seed(SEED); np.random.seed(SEED); tf.random.set_seed(SEED)

print("TensorFlow:", tf.__version__)
print("GPU:", tf.config.list_physical_devices("GPU"))

try:
    tf.keras.mixed_precision.set_global_policy("mixed_float16")
    print("Mixed precision enabled")
except Exception as e:
    print("Mixed precision not enabled:", e)

In [ ]:
# 2) Upload the ORIGINAL dataset ZIP directly from Windows
from google.colab import files

uploaded = files.upload()
zip_names = [n for n in uploaded if n.lower().endswith(".zip")]
if not zip_names:
    raise ValueError("Upload the original flower dataset ZIP.")

ZIP_PATH = Path("/content") / zip_names[0]
EXTRACT_DIR = Path("/content/flower_dataset")
EXTRACT_DIR.mkdir(parents=True, exist_ok=True)

with zipfile.ZipFile(ZIP_PATH, "r") as z:
    z.extractall(EXTRACT_DIR)

print("Uploaded:", ZIP_PATH.name)
print("Extracted:", EXTRACT_DIR)

In [ ]:
# 3) Find train/validation folders automatically
VALID_EXTS = {".jpg",".jpeg",".png",".bmp",".webp"}

train_dirs = [p for p in EXTRACT_DIR.rglob("train") if p.is_dir()]
valid_dirs = [p for p in EXTRACT_DIR.rglob("validation") if p.is_dir()]
valid_dirs += [p for p in EXTRACT_DIR.rglob("valid") if p.is_dir()]

TRAIN_ROOT = train_dirs[0] if train_dirs else None
VALID_ROOT = valid_dirs[0] if valid_dirs else None

print("TRAIN_ROOT:", TRAIN_ROOT)
print("VALID_ROOT:", VALID_ROOT)

if TRAIN_ROOT is None and VALID_ROOT is None:
    raise FileNotFoundError("No train/validation folder found in the uploaded ZIP.")

In [ ]:
# 4) Build a complete FILE-LEVEL image table
EXPECTED_CLASSES = [
    "bluebell","buttercup","colts_foot","cowslip","crocus","daffodil",
    "daisy","dandelion","fritillary","iris","lily_valley","pansy",
    "snowdrop","sunflower","tigerlily","tulip","windflower"
]

records = []
for root in [TRAIN_ROOT, VALID_ROOT]:
    if root is None: continue
    for cls_dir in root.iterdir():
        if not cls_dir.is_dir(): continue
        for p in cls_dir.rglob("*"):
            if p.is_file() and p.suffix.lower() in VALID_EXTS:
                records.append((str(p), cls_dir.name))

df = pd.DataFrame(records, columns=["filepath","class_name"])
found = sorted(df.class_name.unique())

print("Total images:", len(df))
print("Found classes:", found)

missing = sorted(set(EXPECTED_CLASSES) - set(found))
extra = sorted(set(found) - set(EXPECTED_CLASSES))
if missing: raise ValueError(f"Missing classes: {missing}")
if extra: raise ValueError(f"Unexpected classes: {extra}")

label_map = {c:i for i,c in enumerate(EXPECTED_CLASSES)}
df["label"] = df["class_name"].map(label_map)

display(df["class_name"].value_counts().sort_index())

In [ ]:
# 5) TRUE STRATIFIED 80/10/10 split
train_df, temp_df = train_test_split(
    df, test_size=0.20, stratify=df["label"], random_state=SEED
)
val_df, test_df = train_test_split(
    temp_df, test_size=0.50, stratify=temp_df["label"], random_state=SEED
)

train_df = train_df.sample(frac=1, random_state=SEED).reset_index(drop=True)
val_df = val_df.sample(frac=1, random_state=SEED).reset_index(drop=True)
test_df = test_df.sample(frac=1, random_state=SEED).reset_index(drop=True)

print(f"Train: {len(train_df)} ({len(train_df)/len(df):.2%})")
print(f"Val:   {len(val_df)} ({len(val_df)/len(df):.2%})")
print(f"Test:  {len(test_df)} ({len(test_df)/len(df):.2%})")

dist = pd.DataFrame({
    "train": train_df.class_name.value_counts(),
    "validation": val_df.class_name.value_counts(),
    "test": test_df.class_name.value_counts()
}).fillna(0).astype(int).sort_index()
display(dist)

assert all(set(x.class_name)==set(EXPECTED_CLASSES) for x in [train_df,val_df,test_df])

In [ ]:
# 6) TensorFlow input pipeline
IMG_SIZE = (260,260)
BATCH_SIZE = 32
AUTOTUNE = tf.data.AUTOTUNE
NUM_CLASSES = 17

def make_dataset(frame, shuffle=False):
    paths = frame.filepath.values
    labels = frame.label.values.astype(np.int32)
    ds = tf.data.Dataset.from_tensor_slices((paths, labels))

    def load(path, label):
        img = tf.io.read_file(path)
        img = tf.io.decode_image(img, channels=3, expand_animations=False)
        img.set_shape([None,None,3])
        img = tf.image.resize(img, IMG_SIZE)
        return tf.cast(img, tf.float32), label

    ds = ds.map(load, num_parallel_calls=AUTOTUNE)
    if shuffle:
        ds = ds.shuffle(len(frame), seed=SEED, reshuffle_each_iteration=True)
    return ds.batch(BATCH_SIZE).prefetch(AUTOTUNE)

train_ds = make_dataset(train_df, True)
val_ds = make_dataset(val_df)
test_ds = make_dataset(test_df)

weights = compute_class_weight(
    class_weight="balanced",
    classes=np.arange(NUM_CLASSES),
    y=train_df.label.values
)
CLASS_WEIGHTS = {i:float(w) for i,w in enumerate(weights)}

print("Datasets ready.")

In [ ]:
# 7) Augmentation
data_augmentation = tf.keras.Sequential([
    tf.keras.layers.RandomFlip("horizontal"),
    tf.keras.layers.RandomRotation(0.10),
    tf.keras.layers.RandomZoom(0.15),
    tf.keras.layers.RandomTranslation(0.08,0.08),
    tf.keras.layers.RandomContrast(0.12),
], name="flower_augmentation")

In [ ]:
# 8) Strong CNN: EfficientNetB2
from tensorflow.keras import layers, Model
from tensorflow.keras.applications import EfficientNetB2

base = EfficientNetB2(
    include_top=False, weights="imagenet",
    input_shape=(*IMG_SIZE,3)
)
base.trainable = False

inputs = layers.Input(shape=(*IMG_SIZE,3), name="image")
x = data_augmentation(inputs)
x = base(x, training=False)
x = layers.GlobalAveragePooling2D()(x)
x = layers.BatchNormalization()(x)
x = layers.Dropout(0.35)(x)
x = layers.Dense(256, activation="swish")(x)
x = layers.Dropout(0.25)(x)
outputs = layers.Dense(NUM_CLASSES, activation="softmax", dtype="float32")(x)

model = Model(inputs, outputs, name="Flower17_EfficientNetB2")

model.compile(
    optimizer=tf.keras.optimizers.AdamW(learning_rate=1e-3, weight_decay=1e-4),
    loss=tf.keras.losses.SparseCategoricalCrossentropy(label_smoothing=0.05),
    metrics=["accuracy"]
)
model.summary()

In [ ]:
# 9) Stage 1: train classifier head
MODEL_DIR = Path("/content/flower_models")
MODEL_DIR.mkdir(exist_ok=True)

from tensorflow.keras.callbacks import EarlyStopping, ModelCheckpoint, ReduceLROnPlateau

cb1 = [
    ModelCheckpoint(MODEL_DIR/"best_stage1.keras", monitor="val_accuracy",
                    mode="max", save_best_only=True, verbose=1),
    EarlyStopping(monitor="val_accuracy", mode="max", patience=8,
                   restore_best_weights=True, verbose=1),
    ReduceLROnPlateau(monitor="val_loss", factor=0.3, patience=3,
                      min_lr=1e-7, verbose=1)
]

history1 = model.fit(
    train_ds, validation_data=val_ds, epochs=25,
    class_weight=CLASS_WEIGHTS, callbacks=cb1
)

In [ ]:
# 10) Stage 2: fine-tune upper CNN layers
base.trainable = True
freeze_until = int(len(base.layers)*0.60)

for layer in base.layers[:freeze_until]:
    layer.trainable = False
for layer in base.layers:
    if isinstance(layer, tf.keras.layers.BatchNormalization):
        layer.trainable = False

model.compile(
    optimizer=tf.keras.optimizers.AdamW(learning_rate=2e-5, weight_decay=1e-5),
    loss=tf.keras.losses.SparseCategoricalCrossentropy(label_smoothing=0.03),
    metrics=["accuracy"]
)

cb2 = [
    ModelCheckpoint(MODEL_DIR/"best_finetuned.keras", monitor="val_accuracy",
                    mode="max", save_best_only=True, verbose=1),
    EarlyStopping(monitor="val_accuracy", mode="max", patience=10,
                  restore_best_weights=True, verbose=1),
    ReduceLROnPlateau(monitor="val_loss", factor=0.3, patience=3,
                      min_lr=1e-7, verbose=1)
]

history2 = model.fit(
    train_ds, validation_data=val_ds, epochs=35,
    class_weight=CLASS_WEIGHTS, callbacks=cb2
)

In [ ]:
# 11) FINAL test evaluation
test_loss, test_acc = model.evaluate(test_ds, verbose=1)
print(f"TEST ACCURACY: {test_acc*100:.2f}%")
print(f"TEST LOSS: {test_loss:.4f}")

In [ ]:
# 12) Per-class report + confusion matrix
y_true = np.concatenate([y.numpy() for _,y in test_ds])
probs = model.predict(test_ds, verbose=1)
y_pred = np.argmax(probs, axis=1)

print(classification_report(
    y_true, y_pred, target_names=EXPECTED_CLASSES, digits=4
))

cm = confusion_matrix(y_true, y_pred)
plt.figure(figsize=(14,12))
plt.imshow(cm, interpolation="nearest")
plt.title("17-Class Confusion Matrix")
plt.colorbar()
plt.xticks(range(NUM_CLASSES), EXPECTED_CLASSES, rotation=90)
plt.yticks(range(NUM_CLASSES), EXPECTED_CLASSES)
plt.xlabel("Predicted")
plt.ylabel("True")
for i in range(NUM_CLASSES):
    for j in range(NUM_CLASSES):
        plt.text(j,i,cm[i,j],ha="center",va="center")
plt.tight_layout()
plt.show()

In [ ]:
# 13) Save production model + exact class mapping
FINAL_MODEL = MODEL_DIR/"final_model.keras"
CLASS_FILE = MODEL_DIR/"class_names.json"

model.save(FINAL_MODEL)
with open(CLASS_FILE,"w",encoding="utf-8") as f:
    json.dump(EXPECTED_CLASSES,f,indent=2)

print(FINAL_MODEL)
print(CLASS_FILE)

In [ ]:
# 14) Single-image production prediction
def predict_flower(image_path, confidence_threshold=0.45):
    img = Image.open(image_path).convert("RGB").resize(IMG_SIZE)
    arr = np.asarray(img, dtype=np.float32)[None,...]
    probabilities = model.predict(arr, verbose=0)[0]

    idx = int(np.argmax(probabilities))
    confidence = float(probabilities[idx])

    return {
        "class_name": EXPECTED_CLASSES[idx],
        "confidence": confidence,
        "confidence_percent": confidence*100,
        "low_confidence": confidence < confidence_threshold
    }

# Example:
# print(predict_flower("/content/my_flower.jpg"))

In [ ]:
# 15) Top-5 diagnostic (deployment still returns ONE final class)
def top5_predictions(image_path):
    img = Image.open(image_path).convert("RGB").resize(IMG_SIZE)
    arr = np.asarray(img, dtype=np.float32)[None,...]
    p = model.predict(arr, verbose=0)[0]
    ids = np.argsort(p)[::-1][:5]
    return [(EXPECTED_CLASSES[i], float(p[i])) for i in ids]

In [ ]:
# 16) Download trained model + labels to Windows
from google.colab import files
files.download(str(FINAL_MODEL))
files.download(str(CLASS_FILE))

In [ ]:
# 17) Optional deployment bundle
DEPLOY_DIR = Path("/content/Flower17_Deployment")
DEPLOY_DIR.mkdir(exist_ok=True)

shutil.copy2(FINAL_MODEL, DEPLOY_DIR/"final_model.keras")
shutil.copy2(CLASS_FILE, DEPLOY_DIR/"class_names.json")
train_df.to_csv(DEPLOY_DIR/"train_split.csv", index=False)
val_df.to_csv(DEPLOY_DIR/"validation_split.csv", index=False)
test_df.to_csv(DEPLOY_DIR/"test_split.csv", index=False)

bundle = shutil.make_archive("/content/Flower17_Deployment","zip",DEPLOY_DIR)
print(bundle)
files.download(bundle)